# Data Cleaning -- Training Dataset
**This notebook prepares Flatiron Health data for patients with advanced urothelial cancer in anticipation of training a gradient-boosted survival model to predict mortality from the start of first-line treatment. Specifically, it processes and cleans the training cohort. Prior to data cleaning, the dataset is randomly split into training (80%) and testing (20%) subsets.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorUrothelial, merge_dataframes

## Split into train and test sets

In [2]:
df = pd.read_csv('../data/LineOfTherapy.csv')

In [3]:
df = (
    df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [4]:
df.shape

(9382, 2)

In [5]:
processor = DataProcessorUrothelial()

In [6]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvUrothelial_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvUrothelial_Progression.csv',
                                           drop_dates = False)

2025-12-11 13:30:01,574 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (9040, 2) and unique PatientIDs: 9040
2025-12-11 13:30:01,585 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (9382, 3) and unique PatientIDs: 9382
2025-12-11 13:30:01,990 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2025-12-11 13:30:01,997 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (9382, 6) and unique PatientIDs: 9382. There are 0 out of 9382 patients with missing duration values


In [7]:
df = pd.merge(df, mortality_df[['PatientID', 'event']], on = 'PatientID', how = 'left')

In [8]:
df.shape

(9382, 3)

In [9]:
# Stratify split on event 
train, test = train_test_split(
    df,
    test_size = 0.2,
    stratify = df['event'],  
    random_state = 42
)

In [10]:
train.shape

(7505, 3)

In [11]:
test.shape

(1877, 3)

In [12]:
train[['PatientID']].to_csv('../outputs/train_patient_ids.csv', index = False)
test[['PatientID']].to_csv('../outputs/test_patient_ids.csv', index = False)

## Clean CSV files 

In [13]:
train_ids = pd.read_csv('../outputs/train_patient_ids.csv')

In [14]:
train_ids = train_ids.PatientID.to_list()

In [15]:
index_date_df = df[df.PatientID.isin(train_ids)]

In [16]:
index_date_df.shape

(7505, 3)

In [17]:
index_date_df = index_date_df[['PatientID', 'StartDate']]

### Process Enhanced_AdvUrothelial.csv

In [18]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvUrothelial.csv',
                                         index_date_df = index_date_df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2025-12-11 13:30:02,083 - INFO - Successfully read Enhanced_AdvUrothelial.csv file with shape: (13129, 13) and unique PatientIDs: 13129
2025-12-11 13:30:02,089 - INFO - Successfully filtered Enhanced_AdvUrothelial.csv file with shape: (7505, 14) and unique PatientIDs: 7505
2025-12-11 13:30:02,108 - INFO - Successfully processed Enhanced_AdvUrothelial.csv file with final shape: (7505, 16) and unique PatientIDs: 7505


In [19]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])

In [20]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [21]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
unknown    3509
IV         2576
III         685
II          570
I           126
0            39
Name: count, dtype: int64

In [22]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [23]:
enhanced_df['PrimarySite_lower'] = enhanced_df['PrimarySite'].isin(['Bladder', 'Urethra']).astype('int64')
enhanced_df = enhanced_df.drop(columns = ['PrimarySite'])

In [24]:
# drop dates
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate',
                                          'AdvancedDiagnosisDate',
                                          'SurgeryDate',
                                          'imported_StartDate'])

### Process Demographics.csv 

In [25]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2025-12-11 13:30:02,148 - INFO - Successfully read Demographics.csv file with shape: (13129, 6) and unique PatientIDs: 13129
2025-12-11 13:30:02,161 - INFO - Successfully processed Demographics.csv file with final shape: (7505, 6) and unique PatientIDs: 7505


In [26]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      5500
F      2003
NaN       2
Name: count, dtype: int64

In [27]:
# Impute missing with male
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [28]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvUrothelialBiomarkers.csv

In [29]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2025-12-11 13:30:02,192 - INFO - Successfully read Enhanced_AdvUrothelialBiomarkers.csv file with shape: (9924, 19) and unique PatientIDs: 4251
2025-12-11 13:30:02,203 - INFO - Successfully merged Enhanced_AdvUrothelialBiomarkers.csv df with index_date_df resulting in shape: (6917, 20) and unique PatientIDs: 2891
2025-12-11 13:30:02,255 - INFO - Successfully processed Enhanced_AdvUrothelialBiomarkers.csv file with final shape: (7505, 4) and unique PatientIDs: 7505


In [30]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [31]:
biomarkers_df.PDL1_binary.value_counts(dropna = False, normalize = True)

PDL1_binary
NaN     0.986676
>=1%    0.013324
Name: proportion, dtype: float64

In [32]:
biomarkers_df = biomarkers_df.drop(columns = ['PDL1_percent_staining'])

In [33]:
biomarkers_df['FGFR_status'] = biomarkers_df['FGFR_status'].fillna('unknown')
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [34]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2025-12-11 13:30:02,336 - INFO - Successfully read ECOG.csv file with shape: (184794, 4) and unique PatientIDs: 9933
2025-12-11 13:30:02,376 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (135622, 5) and unique PatientIDs: 6178
2025-12-11 13:30:02,473 - INFO - Successfully processed ECOG.csv file with final shape: (7505, 3) and unique PatientIDs: 7505


In [35]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    2478
1      2239
0      1765
2       792
3       214
4        17
Name: count, dtype: int64

In [36]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [37]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [38]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-12-11 13:30:06,027 - INFO - Successfully read Vitals.csv file with shape: (3604484, 16) and unique PatientIDs: 13109
2025-12-11 13:30:07,749 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (2434144, 17) and unique PatientIDs: 7505
2025-12-11 13:30:08,826 - INFO - Successfully processed Vitals.csv file with final shape: (7505, 8) and unique PatientIDs: 7505


### Process Lab.csv

In [39]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2025-12-11 13:30:20,997 - INFO - Successfully read Lab.csv file with shape: (9373598, 17) and unique PatientIDs: 12700
2025-12-11 13:30:24,830 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (6621642, 18) and unique PatientIDs: 7445
2025-12-11 13:30:38,870 - INFO - Successfully processed Lab.csv file with final shape: (7505, 76) and unique PatientIDs: 7505


### Process MedicationAdministration.csv

In [40]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2025-12-11 13:30:40,161 - INFO - Successfully read MedicationAdministration.csv file with shape: (997836, 11) and unique PatientIDs: 10983
2025-12-11 13:30:40,529 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (673392, 12) and unique PatientIDs: 7332
2025-12-11 13:30:40,588 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (7505, 9) and unique PatientIDs: 7505


### Process Diagnosis.csv

In [41]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2025-12-11 13:30:41,016 - INFO - Successfully read Diagnosis.csv file with shape: (625348, 6) and unique PatientIDs: 13129
2025-12-11 13:30:41,146 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (409535, 7) and unique PatientIDs: 7505
2025-12-11 13:30:42,233 - INFO - Successfully processed Diagnosis.csv file with final shape: (7505, 40) and unique PatientIDs: 7505


### Process Enhanced_Mortality_V2.csv

In [42]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvUrothelial_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvUrothelial_Progression.csv',
                                           drop_dates = False)

2025-12-11 13:30:42,251 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (9040, 2) and unique PatientIDs: 9040
2025-12-11 13:30:42,262 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (7505, 3) and unique PatientIDs: 7505
2025-12-11 13:30:42,654 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2025-12-11 13:30:42,660 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (7505, 6) and unique PatientIDs: 7505. There are 0 out of 7505 patients with missing duration values


In [43]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [44]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [45]:
training_df = merge_dataframes(enhanced_df,
                               demographics_df,
                               biomarkers_df,
                               ecog_df,
                               vitals_df,
                               labs_df,
                               medications_df,
                               diagnosis_df, 
                               mortality_df,
                               merge_type = 'inner')

2025-12-11 13:30:42,682 - INFO - Anticipated number of merges: 8
2025-12-11 13:30:42,682 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 156
2025-12-11 13:30:42,683 - INFO - Dataset 1 shape: (7505, 14), unique PatientIDs: 7505
2025-12-11 13:30:42,684 - INFO - Dataset 2 shape: (7505, 6), unique PatientIDs: 7505
2025-12-11 13:30:42,685 - INFO - Dataset 3 shape: (7505, 4), unique PatientIDs: 7505
2025-12-11 13:30:42,686 - INFO - Dataset 4 shape: (7505, 4), unique PatientIDs: 7505
2025-12-11 13:30:42,686 - INFO - Dataset 5 shape: (7505, 8), unique PatientIDs: 7505
2025-12-11 13:30:42,687 - INFO - Dataset 6 shape: (7505, 76), unique PatientIDs: 7505
2025-12-11 13:30:42,689 - INFO - Dataset 7 shape: (7505, 9), unique PatientIDs: 7505
2025-12-11 13:30:42,690 - INFO - Dataset 8 shape: (7505, 40), unique PatientIDs: 7505
2025-12-11 13:30:42,691 - INFO - Dataset 9 shape: (7487, 3), unique PatientIDs: 7487
2025-12-11 13:30:42,694 - 

In [46]:
training_df.shape

(7487, 156)

## Export dataframe

In [47]:
training_df.to_csv('../outputs/1L_features_training.csv', index = False)

In [48]:
# Save dtypes
training_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_training_dtypes.csv')